In [12]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    # Mount Google Drive to persist the datasets and cloned repository
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing dependencies...")
    os.system('pip install -q pandas scikit-learn fasttext huggingface_hub gdown')
    print("Setup complete!")


In [13]:
input_file = 'datasets/finetuning/train.csv'
benchmark_dir = 'datasets/preprocessed'
output_dir = 'models/finetuned/NLLB_LID_218'


In [14]:
import os
# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

import json
import glob
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score
import fasttext
from huggingface_hub import hf_hub_download

MODEL_NAME = "NLLB LID-218"
MODEL_ID = "nllb_lid_218"


In [15]:
print("Downloading NLLB LID-218 base model from Hugging Face (~1.18GB)...")
base_model_path = hf_hub_download(repo_id="facebook/fasttext-language-identification", filename="model.bin")
base_model_dir = os.path.dirname(base_model_path)

print(f"Loading base NLLB LID-218 model from {base_model_path}...")
base_model = fasttext.load_model(base_model_path)

# Extract pretrained word vectors for transfer learning / fine-tuning
vec_file_path = os.path.join(base_model_dir, "nllb_lid_218.vec")
if not os.path.exists(vec_file_path):
    print("Extracting pre-trained word vectors to .vec format...")
    words = base_model.get_words()
    dim = base_model.get_dimension()
    with open(vec_file_path, "w", encoding="utf-8") as f:
        f.write(f"{len(words)} {dim}\n")
        for word in words:
            v_str = " ".join(map(str, base_model.get_word_vector(word)))
            f.write(f"{word} {v_str}\n")
    print(f"Extracted {len(words)} vectors into {vec_file_path}")


Loading base NLLB LID-218 model from C:\Users\USER\.cache\huggingface\hub\models--facebook--fasttext-language-identification\snapshots\3af127d4124fc58b75666f3594bb5143b9757e78\model.bin...


In [16]:
print(f"Loading finetuning dataset from {input_file}...")
df = pd.read_csv(input_file)
print(f"Loaded {len(df)} rows.")

def format_text(text):
    return str(text).replace("\n", " ").strip()

df["clean_text"] = df["text"].apply(format_text)
df["ft_line"] = "__label__" + df["label"].astype(str) + " " + df["clean_text"]

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df["label"])

os.makedirs(output_dir, exist_ok=True)
train_ft_file = os.path.join(output_dir, "train_formatted.txt")
val_ft_file = os.path.join(output_dir, "val_formatted.txt")

with open(train_ft_file, "w", encoding="utf-8") as f:
    f.write("\n".join(train_df["ft_line"].tolist()) + "\n")

with open(val_ft_file, "w", encoding="utf-8") as f:
    f.write("\n".join(val_df["ft_line"].tolist()) + "\n")

print(f"Saved {len(train_df)} training samples to {train_ft_file}")
print(f"Saved {len(val_df)} validation samples to {val_ft_file}")


Loading finetuning dataset from datasets/finetuning/train.csv...
Loaded 60285 rows.
Saved 54256 training samples to models/finetuned/NLLB_LID_218\train_formatted.txt
Saved 6029 validation samples to models/finetuned/NLLB_LID_218\val_formatted.txt


In [17]:
print(f"Starting fine-tuning for {MODEL_NAME}...")
finetuned_model = fasttext.train_supervised(
    input=train_ft_file,
    pretrainedVectors=vec_file_path,
    dim=base_model.get_dimension(),
    epoch=25,
    lr=0.5,
    wordNgrams=2,
    loss="softmax"
)

save_model_path = os.path.join(output_dir, f"{MODEL_ID}_finetuned.bin")
finetuned_model.save_model(save_model_path)
print(f"Fine-tuned model successfully saved to {save_model_path}")


Starting fine-tuning for NLLB LID-218...
Fine-tuned model successfully saved to models/finetuned/NLLB_LID_218\nllb_lid_218_finetuned.bin


In [18]:
val_texts = val_df["clean_text"].tolist()
val_true = val_df["label"].astype(str).tolist()

print(f"Evaluating fine-tuned {MODEL_NAME} on {len(val_texts)} validation samples...")
preds, _ = finetuned_model.predict(val_texts, k=1)
val_pred = [p[0].replace("__label__", "") for p in preds]

acc = accuracy_score(val_true, val_pred)
macro_f1 = f1_score(val_true, val_pred, average="macro")

print("\n" + "=" * 48)
print(f"VALIDATION FINE-TUNING RESULTS ({MODEL_NAME})")
print("=" * 48)
print(f"Accuracy:  {acc * 100:.2f}%")
print(f"Macro F1:  {macro_f1 * 100:.2f}%")
print("=" * 48)
print("\nPer-language breakdown:\n")
print(classification_report(val_true, val_pred, digits=4, zero_division=0))


Evaluating fine-tuned NLLB LID-218 on 6029 validation samples...

VALIDATION FINE-TUNING RESULTS (NLLB LID-218)
Accuracy:  98.56%
Macro F1:  98.52%

Per-language breakdown:

              precision    recall  f1-score   support

        pali     0.9888    0.9813    0.9850      2349
    sanskrit     0.9940    0.9743    0.9840      1012
     sinhala     0.9797    0.9936    0.9866      2668

    accuracy                         0.9856      6029
   macro avg     0.9875    0.9831    0.9852      6029
weighted avg     0.9856    0.9856    0.9856      6029



In [19]:
print(f"Evaluating fine-tuned {MODEL_NAME} on Sinhala script target languages across benchmark datasets in {benchmark_dir}...")

TARGET_LANGUAGES = ["sinhala", "pali", "sanskrit"]

def map_benchmark_label(row):
    lbl = row.get("label")
    src = row.get("source")
    if lbl in ["sin", "sin_Sinh", "sinhala", "si"]:
        return "sinhala"
    if lbl in ["pli", "pli_Sinh", "pali", "pi"]:
        return "pali"
    if lbl in ["san_Sinh", "sanskrit"] or (lbl == "san" and src in ["DCS", "SansinNT", "SiDiaC-v2"]):
        return "sanskrit"
    return None

def load_benchmark_dataset(file_path):
    records = []
    with open(file_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            mapped_label = map_benchmark_label(row)
            if mapped_label:
                row["target_label"] = mapped_label
                records.append(row)
    return pd.DataFrame(records)

benchmark_files = sorted(glob.glob(os.path.join(benchmark_dir, "*.jsonl")))
if not benchmark_files:
    print(f"No benchmark datasets found in {benchmark_dir}.")
else:
    results_dir = os.path.join("datasets", "benchmark_results")
    os.makedirs(results_dir, exist_ok=True)
    
    for file_path in benchmark_files:
        dataset_name = os.path.splitext(os.path.basename(file_path))[0]
        df_bench = load_benchmark_dataset(file_path)
        if df_bench.empty:
            print(f"No matching target languages found in {dataset_name}.")
            continue
        
        texts = df_bench["text"].apply(format_text).tolist()
        print(f"\nEvaluating {len(texts)} target language samples from {dataset_name}...")
        preds, _ = finetuned_model.predict(texts, k=1)
        
        results = df_bench[["text", "label", "source"]].copy()
        results["true_label"] = df_bench["target_label"]
        results["predicted_label"] = [p[0].replace("__label__", "") for p in preds]
        
        acc_b = accuracy_score(results["true_label"], results["predicted_label"])
        macro_f1_b = f1_score(
            results["true_label"], results["predicted_label"],
            average="macro", labels=TARGET_LANGUAGES, zero_division=0
        )
        
        print("=" * 65)
        print(f"BENCHMARK RESULTS ({MODEL_NAME} Finetuned on train.csv - Evaluated on {dataset_name})")
        print("=" * 65)
        print(f"Accuracy:  {acc_b * 100:.2f}%")
        print(f"Macro F1:  {macro_f1_b * 100:.2f}%")
        print("=" * 65)
        print("\nPer-language breakdown (F1 scores & metrics for target languages):\n")
        print(classification_report(
            results["true_label"], results["predicted_label"],
            labels=TARGET_LANGUAGES, digits=4, zero_division=0
        ))
        
        out_csv = os.path.join(results_dir, f"{MODEL_ID}_finetuned_{dataset_name}.csv")
        results.to_csv(out_csv, index=False)
        print(f"Saved benchmark predictions to {out_csv}")


Evaluating fine-tuned NLLB LID-218 on Sinhala script target languages across benchmark datasets in datasets/preprocessed...

Evaluating 7047 target language samples from commonlid...
BENCHMARK RESULTS (NLLB LID-218 Finetuned on train.csv - Evaluated on commonlid)
Accuracy:  97.79%
Macro F1:  97.84%

Per-language breakdown (F1 scores & metrics for target languages):

              precision    recall  f1-score   support

     sinhala     0.9542    0.9978    0.9755      2693
        pali     0.9929    0.9647    0.9786      3027
    sanskrit     0.9953    0.9676    0.9813      1327

    accuracy                         0.9779      7047
   macro avg     0.9808    0.9767    0.9784      7047
weighted avg     0.9786    0.9779    0.9779      7047

Saved benchmark predictions to datasets\benchmark_results\nllb_lid_218_finetuned_commonlid.csv

Evaluating 7047 target language samples from flores_plus...
BENCHMARK RESULTS (NLLB LID-218 Finetuned on train.csv - Evaluated on flores_plus)
Accuracy:  

In [20]:
print(f"Evaluating fine-tuned {MODEL_NAME} across ALL benchmark languages in {benchmark_dir}...")

ALL_BENCHMARK_LANGUAGES = [
    "sinhala", "pali", "sanskrit", "sanskrit_deva", "english", "tamil",
    "hindi", "bengali", "arabic", "french", "german"
]

LABEL_MAPPING_ALL = {
    "sin": "sinhala", "sin_Sinh": "sinhala", "sinhala": "sinhala", "si": "sinhala",
    "pli": "pali", "pli_Sinh": "pali", "pli_Latn": "pali", "pali": "pali", "pi": "pali",
    "san_Sinh": "sanskrit",
    "san_Deva": "sanskrit_deva", "sa": "sanskrit_deva",
    "eng": "english", "eng_Latn": "english", "english": "english", "en": "english",
    "tam": "tamil", "tam_Taml": "tamil", "tamil": "tamil", "ta": "tamil",
    "hin": "hindi", "hin_Deva": "hindi", "hindi": "hindi", "hi": "hindi",
    "ben": "bengali", "ben_Beng": "bengali", "bengali": "bengali", "bn": "bengali",
    "arb": "arabic", "arb_Arab": "arabic", "arabic": "arabic", "ar": "arabic",
    "fra": "french", "fra_Latn": "french", "french": "french", "fr": "french",
    "deu": "german", "deu_Latn": "german", "german": "german", "de": "german"
}

def map_all_label(row):
    lbl = row.get("label")
    src = row.get("source")
    if lbl == "san":
        if src in ["DCS", "SansinNT", "SiDiaC-v2"]:
            return "sanskrit"
        else:
            return "sanskrit_deva"
    return LABEL_MAPPING_ALL.get(lbl)

def load_all_languages_dataset(file_path):
    records = []
    with open(file_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            mapped_label = map_all_label(row)
            if mapped_label:
                row["target_label"] = mapped_label
                records.append(row)
    return pd.DataFrame(records)

benchmark_files = sorted(glob.glob(os.path.join(benchmark_dir, "*.jsonl")))
if not benchmark_files:
    print(f"No benchmark datasets found in {benchmark_dir}.")
else:
    results_dir = os.path.join("datasets", "benchmark_results")
    os.makedirs(results_dir, exist_ok=True)
    
    for file_path in benchmark_files:
        dataset_name = os.path.splitext(os.path.basename(file_path))[0]
        df_all = load_all_languages_dataset(file_path)
        if df_all.empty:
            print(f"No matching languages found in {dataset_name}.")
            continue
        
        texts = df_all["text"].apply(format_text).tolist()
        print(f"\nEvaluating {len(texts)} samples across ALL benchmark languages from {dataset_name}...")
        preds, _ = finetuned_model.predict(texts, k=1)
        
        results = df_all[["text", "label", "source"]].copy()
        results["true_label"] = df_all["target_label"]
        results["predicted_label"] = [p[0].replace("__label__", "") for p in preds]
        
        acc_all = accuracy_score(results["true_label"], results["predicted_label"])
        macro_f1_all = f1_score(
            results["true_label"], results["predicted_label"],
            average="macro", labels=ALL_BENCHMARK_LANGUAGES, zero_division=0
        )
        
        print("=" * 65)
        print(f"ALL LANGUAGES BENCHMARK RESULTS ({MODEL_NAME} Finetuned on train.csv - Evaluated on {dataset_name})")
        print("=" * 65)
        print(f"Accuracy:  {acc_all * 100:.2f}%")
        print(f"Macro F1:  {macro_f1_all * 100:.2f}%")
        print("=" * 65)
        print("\nPer-language breakdown (All Benchmark Languages):\n")
        print(classification_report(
            results["true_label"], results["predicted_label"],
            labels=ALL_BENCHMARK_LANGUAGES, digits=4, zero_division=0
        ))
        
        out_csv = os.path.join(results_dir, f"{MODEL_ID}_finetuned_all_langs_{dataset_name}.csv")
        results.to_csv(out_csv, index=False)
        print(f"Saved all-languages benchmark predictions to {out_csv}")


Evaluating fine-tuned NLLB LID-218 across ALL benchmark languages in datasets/preprocessed...

Evaluating 77974 samples across ALL benchmark languages from commonlid...
ALL LANGUAGES BENCHMARK RESULTS (NLLB LID-218 Finetuned on train.csv - Evaluated on commonlid)
Accuracy:  8.84%
Macro F1:  7.09%

Per-language breakdown (All Benchmark Languages):

               precision    recall  f1-score   support

      sinhala     0.0606    0.9978    0.1142      2693
         pali     0.0984    0.9647    0.1786      3027
     sanskrit     0.3254    0.9676    0.4870      1327
sanskrit_deva     0.0000    0.0000    0.0000       895
      english     0.0000    0.0000    0.0000     27461
        tamil     0.0000    0.0000    0.0000        81
        hindi     0.0000    0.0000    0.0000      3666
      bengali     0.0000    0.0000    0.0000      1886
       arabic     0.0000    0.0000    0.0000     26152
       french     0.0000    0.0000    0.0000      3233
       german     0.0000    0.0000    0.0000

In [21]:
print("\n" + "=" * 50)
print(f"FINE-TUNING & BENCHMARKING COMPLETE FOR {MODEL_NAME}")
print(f"Fine-tuned model saved to: {save_model_path}")
print("=" * 50)



FINE-TUNING & BENCHMARKING COMPLETE FOR NLLB LID-218
Fine-tuned model saved to: models/finetuned/NLLB_LID_218\nllb_lid_218_finetuned.bin


### validation test

In [32]:
import os
import json
import glob
import gc
import fasttext
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# -------------------------------------------------
# Paths
# -------------------------------------------------

ORIGINAL_MODEL = r"D:\Projects\ML Projects\LangID - DSE project\models\nllb_lid_218_original.bin"
EXTENDED_MODEL = r"D:\Projects\ML Projects\LangID - DSE project\models\nllb_lid_220_ft_e3_lr001.bin"


# -------------------------------------------------
# We only test EXISTING languages here.
# Pali-Sinh and Sanskrit-Sinh have not been trained yet.
# -------------------------------------------------

OLD_LANGUAGES = [
    "sinhala",
    "sanskrit_deva",
    "english",
    "tamil",
    "hindi",
    "bengali",
    "arabic",
    "french",
    "german"
]


# Benchmark true-label mapping
TRUE_LABEL_MAPPING = {
    "sin": "sinhala",
    "sin_Sinh": "sinhala",
    "sinhala": "sinhala",
    "si": "sinhala",

    "san_Deva": "sanskrit_deva",
    "sa": "sanskrit_deva",

    "eng": "english",
    "eng_Latn": "english",
    "english": "english",
    "en": "english",

    "tam": "tamil",
    "tam_Taml": "tamil",
    "tamil": "tamil",
    "ta": "tamil",

    "hin": "hindi",
    "hin_Deva": "hindi",
    "hindi": "hindi",
    "hi": "hindi",

    "ben": "bengali",
    "ben_Beng": "bengali",
    "bengali": "bengali",
    "bn": "bengali",

    "arb": "arabic",
    "arb_Arab": "arabic",
    "arabic": "arabic",
    "ar": "arabic",

    "fra": "french",
    "fra_Latn": "french",
    "french": "french",
    "fr": "french",

    "deu": "german",
    "deu_Latn": "german",
    "german": "german",
    "de": "german",
}


# NLLB prediction -> our evaluation name
PRED_LABEL_MAPPING = {
    "sin_Sinh": "sinhala",
    "san_Deva": "sanskrit_deva",

    "eng_Latn": "english",
    "tam_Taml": "tamil",
    "hin_Deva": "hindi",
    "ben_Beng": "bengali",
    "arb_Arab": "arabic",
    "fra_Latn": "french",
    "deu_Latn": "german",

    # New labels
    "pli_Sinh": "pali",
    "san_Sinh": "sanskrit",
}


def map_true_label(row):
    label = row.get("label")
    source = row.get("source")

    # "san" in project sources means Sinhala-script Sanskrit,
    # so exclude it from this OLD-language preservation check.
    if label == "san":
        if source in ["DCS", "SansinNT", "SiDiaC-v2"]:
            return None
        return "sanskrit_deva"

    return TRUE_LABEL_MAPPING.get(label)


def load_old_languages(path):

    rows = []

    with open(path, encoding="utf-8") as f:

        for line in f:

            row = json.loads(line)

            mapped = map_true_label(row)

            if mapped in OLD_LANGUAGES:

                row["true_label"] = mapped
                rows.append(row)

    return pd.DataFrame(rows)


def clean_text(text):
    return str(text).replace("\n", " ").strip()


# -------------------------------------------------
# Load benchmark data
# -------------------------------------------------

benchmark_files = sorted(
    glob.glob(os.path.join(BENCHMARK_DIR, "*.jsonl"))
)

datasets = {}

for path in benchmark_files:

    name = os.path.splitext(
        os.path.basename(path)
    )[0]

    df = load_old_languages(path)

    if not df.empty:
        datasets[name] = df


print("Benchmark datasets:")

for name, df in datasets.items():
    print(f"{name}: {len(df)} old-language samples")


# -------------------------------------------------
# Prediction function
# -------------------------------------------------

def run_model(model_path):

    print("\nLoading:", model_path)

    model = fasttext.load_model(model_path)

    outputs = {}

    for name, df in datasets.items():

        texts = [
            clean_text(x)
            for x in df["text"].tolist()
        ]

        predictions, _ = model.predict(
            texts,
            k=1
        )

        raw_predictions = [
            p[0].replace("__label__", "")
            for p in predictions
        ]

        mapped_predictions = [
            PRED_LABEL_MAPPING.get(
                p,
                "other"
            )
            for p in raw_predictions
        ]

        outputs[name] = {
            "raw": raw_predictions,
            "mapped": mapped_predictions
        }

    del model
    gc.collect()

    return outputs


# -------------------------------------------------
# Original 218 model
# -------------------------------------------------

original_predictions = run_model(
    ORIGINAL_MODEL
)

# -------------------------------------------------
# Extended 220 model
# -------------------------------------------------

extended_predictions = run_model(
    EXTENDED_MODEL
)


# -------------------------------------------------
# Compare them
# -------------------------------------------------

for name, df in datasets.items():

    y_true = df["true_label"].tolist()

    old_raw = original_predictions[name]["raw"]
    new_raw = extended_predictions[name]["raw"]

    old_pred = original_predictions[name]["mapped"]
    new_pred = extended_predictions[name]["mapped"]

    agreement = sum(
        a == b
        for a, b in zip(old_raw, new_raw)
    ) / len(old_raw)

    changed = sum(
        a != b
        for a, b in zip(old_raw, new_raw)
    )

    new_label_predictions = sum(
        p in ["pli_Sinh", "san_Sinh"]
        for p in new_raw
    )

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print(
        f"Prediction agreement: "
        f"{agreement * 100:.4f}%"
    )

    print(
        f"Changed predictions: "
        f"{changed} / {len(old_raw)}"
    )

    print(
        f"New labels predicted on old-language samples: "
        f"{new_label_predictions}"
    )

    print("\nORIGINAL 218 MODEL")

    print(
        "Accuracy:",
        accuracy_score(y_true, old_pred)
    )

    print(
        "Macro F1:",
        f1_score(
            y_true,
            old_pred,
            labels=OLD_LANGUAGES,
            average="macro",
            zero_division=0
        )
    )

    print("\nEXTENDED 220 MODEL")

    print(
        "Accuracy:",
        accuracy_score(y_true, new_pred)
    )

    print(
        "Macro F1:",
        f1_score(
            y_true,
            new_pred,
            labels=OLD_LANGUAGES,
            average="macro",
            zero_division=0
        )
    )

    print("\nEXTENDED MODEL PER-LANGUAGE RESULTS")

    print(
        classification_report(
            y_true,
            new_pred,
            labels=OLD_LANGUAGES,
            digits=4,
            zero_division=0
        )
    )

Benchmark datasets:
commonlid: 73620 old-language samples
flores_plus: 11801 old-language samples
wili-2018: 9693 old-language samples

Loading: D:\Projects\ML Projects\LangID - DSE project\models\nllb_lid_218_original.bin

Loading: D:\Projects\ML Projects\LangID - DSE project\models\nllb_lid_220_ft_e3_lr001.bin

commonlid
Prediction agreement: 97.5238%
Changed predictions: 1823 / 73620
New labels predicted on old-language samples: 23

ORIGINAL 218 MODEL
Accuracy: 0.9287421896223852
Macro F1: 0.9586237916201521

EXTENDED 220 MODEL
Accuracy: 0.9160554197229014
Macro F1: 0.9563823607297223

EXTENDED MODEL PER-LANGUAGE RESULTS
               precision    recall  f1-score   support

      sinhala     0.9721    0.9955    0.9837      2693
sanskrit_deva     0.9767    0.8894    0.9310       895
      english     0.9944    0.8341    0.9072     27461
        tamil     1.0000    0.9753    0.9875        81
        hindi     0.9930    0.9269    0.9588      3666
      bengali     0.9989    0.9761   

In [24]:
import pandas as pd

df = pd.read_csv("datasets/finetuning/train.csv")

print(df.columns)
print(df.head())
print(df["label"].value_counts())

Index(['id', 'text', 'label', 'source', 'subcorpus', 'group_id'], dtype='str')
   id                                text label     source     subcorpus  \
0   0                       අධිමාසංදීපනි.  pali  SiDiaC-v2  අධිමාස දීපනය   
1   1  නමො අද්වයවාදිනො සම්මා සම්බුද්ධස්ස.  pali  SiDiaC-v2  අධිමාස දීපනය   
2   2       1 නමාමී බුද්ධං චතුසච්ච බුද්ධං  pali  SiDiaC-v2  අධිමාස දීපනය   
3   3         නමාමි ධම්මං අධිමොක්ඛ ධම්මං,  pali  SiDiaC-v2  අධිමාස දීපනය   
4   4               නමාමි සංඝං හතපාප සංඝං  pali  SiDiaC-v2  අධිමාස දීපනය   

               group_id  
0  sidiac2_අධිමාස දීපනය  
1  sidiac2_අධිමාස දීපනය  
2  sidiac2_අධිමාස දීපනය  
3  sidiac2_අධිමාස දීපනය  
4  sidiac2_අධිමාස දීපනය  
label
sinhala     26675
pali        23490
sanskrit    10120
Name: count, dtype: int64


In [25]:
import pandas as pd

df = pd.read_csv("datasets/finetuning/train.csv")

label_map = {
    "sinhala": "__label__sin_Sinh",
    "sin": "__label__sin_Sinh",

    "pali": "__label__pli_Sinh",
    "pli": "__label__pli_Sinh",

    "sanskrit": "__label__san_Sinh",
    "san": "__label__san_Sinh",
}

df["ft_label"] = (
    df["label"]
    .astype(str)
    .str.lower()
    .map(label_map)
)

# Check that every row was mapped
print(df["ft_label"].value_counts(dropna=False))

assert df["ft_label"].notna().all(), "Some labels were not mapped!"

# Clean text so each example is one line
df["clean_text"] = (
    df["text"]
    .astype(str)
    .str.replace("\n", " ", regex=False)
    .str.replace("\r", " ", regex=False)
    .str.strip()
)

output_path = "datasets/finetuning/train_nllb_220.txt"

with open(output_path, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        f.write(f'{row["ft_label"]} {row["clean_text"]}\n')

print("Saved:", output_path)
print("Rows:", len(df))

ft_label
__label__sin_Sinh    26675
__label__pli_Sinh    23490
__label__san_Sinh    10120
Name: count, dtype: int64
Saved: datasets/finetuning/train_nllb_220.txt
Rows: 60285


In [26]:
with open(
    "datasets/finetuning/train_nllb_220.txt",
    encoding="utf-8"
) as f:
    for _ in range(5):
        print(f.readline().strip())

__label__pli_Sinh අධිමාසංදීපනි.
__label__pli_Sinh නමො අද්වයවාදිනො සම්මා සම්බුද්ධස්ස.
__label__pli_Sinh 1 නමාමී බුද්ධං චතුසච්ච බුද්ධං
__label__pli_Sinh නමාමි ධම්මං අධිමොක්ඛ ධම්මං,
__label__pli_Sinh නමාමි සංඝං හතපාප සංඝං


In [27]:
print(df.groupby("ft_label").size())

ft_label
__label__pli_Sinh    23490
__label__san_Sinh    10120
__label__sin_Sinh    26675
dtype: int64


In [31]:
import fasttext
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

MODEL_PATH = r"D:\Projects\ML Projects\LangID - DSE project\models\nllb_lid_220_ft_e3_lr001.bin"

VAL_PATH = r"D:\Projects\ML Projects\LangID - DSE project\data_pipeline\test_dataset_folder\val.csv"

model = fasttext.load_model(MODEL_PATH)
val_df = pd.read_csv(VAL_PATH)

print("Validation size:", len(val_df))
print(val_df["label"].value_counts())


# Clean texts
texts = (
    val_df["text"]
    .astype(str)
    .str.replace("\n", " ", regex=False)
    .str.replace("\r", " ", regex=False)
    .str.strip()
    .tolist()
)


# Predict
predictions, probabilities = model.predict(
    texts,
    k=1
)

raw_predictions = [
    pred[0].replace("__label__", "")
    for pred in predictions
]


# Convert model labels to project labels
pred_map = {
    "sin_Sinh": "sinhala",
    "pli_Sinh": "pali",
    "san_Sinh": "sanskrit",
}

y_pred = [
    pred_map.get(pred, "other")
    for pred in raw_predictions
]

y_true = (
    val_df["label"]
    .astype(str)
    .str.lower()
    .tolist()
)


# Metrics
print("\nAccuracy:")
print(accuracy_score(y_true, y_pred))

print("\nMacro F1:")
print(
    f1_score(
        y_true,
        y_pred,
        labels=["sinhala", "pali", "sanskrit"],
        average="macro",
        zero_division=0
    )
)

print("\nClassification report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=["sinhala", "pali", "sanskrit"],
        digits=4,
        zero_division=0
    )
)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_true,
        y_pred,
        labels=["sinhala", "pali", "sanskrit", "other"]
    )
)

print("\nPredictions outside target classes:")
print(sum(p == "other" for p in y_pred))

print("\nMost common raw predictions:")
print(pd.Series(raw_predictions).value_counts().head(10))

Validation size: 6986
label
sinhala     2895
pali        2800
sanskrit    1291
Name: count, dtype: int64

Accuracy:
0.9627827082736903

Macro F1:
0.9553819940075795

Classification report:
              precision    recall  f1-score   support

     sinhala     0.9706    0.9921    0.9812      2895
        pali     0.9939    0.9379    0.9651      2800
    sanskrit     0.8905    0.9512    0.9199      1291

   micro avg     0.9636    0.9628    0.9632      6986
   macro avg     0.9517    0.9604    0.9554      6986
weighted avg     0.9652    0.9628    0.9634      6986


Confusion matrix:
[[2872   11    7    5]
 [  29 2626  144    1]
 [  58    5 1228    0]
 [   0    0    0    0]]

Predictions outside target classes:
6

Most common raw predictions:
sin_Sinh    2959
pli_Sinh    2642
san_Sinh    1379
deu_Latn       3
hin_Deva       1
npi_Deva       1
vie_Latn       1
Name: count, dtype: int64


### test eval

In [33]:
# ============================================================
# NLLB LID-220 FINETUNED
# ALL-LANGUAGE BENCHMARK EVALUATION
#
# Evaluates on:
#   - CommonLID
#   - FLORES+
#   - WiLI-2018
#
# Languages:
# Sinhala-Sinh, Pali-Sinh, Sanskrit-Sinh, Sanskrit-Deva,
# English, Tamil, Hindi, Bengali, Arabic, French, German
# ============================================================

import os
import json
import glob
import fasttext
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)


# ------------------------------------------------------------
# 1. MODEL
# ------------------------------------------------------------

MODEL_PATH = (
    r"D:\Projects\ML Projects\LangID - DSE project"
    r"\models\nllb_lid_220_ft_e3_lr001.bin"
)

model = fasttext.load_model(MODEL_PATH)

print("Loaded model:", MODEL_PATH)
print("Number of labels:", len(model.get_labels()))


# ------------------------------------------------------------
# 2. BENCHMARK PATHS
# ------------------------------------------------------------

benchmark_input_dir = "datasets/preprocessed"
benchmark_output_dir = "datasets/benchmark_results"

os.makedirs(
    benchmark_output_dir,
    exist_ok=True
)


# ------------------------------------------------------------
# 3. Languages used in your result table
# ------------------------------------------------------------

ALL_BENCHMARK_LANGUAGES = [
    "sinhala",
    "pali",
    "sanskrit",
    "sanskrit_deva",
    "english",
    "tamil",
    "hindi",
    "bengali",
    "arabic",
    "french",
    "german",
]


# ------------------------------------------------------------
# 4. Benchmark label mapping
# ------------------------------------------------------------

LABEL_MAPPING_ALL = {

    # Sinhala-Sinh
    "sin": "sinhala",
    "sin_Sinh": "sinhala",
    "si": "sinhala",
    "sinhala": "sinhala",

    # Pali-Sinh
    "pli": "pali",
    "pli_Sinh": "pali",
    "pali": "pali",

    # English
    "eng": "english",
    "eng_Latn": "english",
    "en": "english",
    "english": "english",

    # Tamil
    "tam": "tamil",
    "tam_Taml": "tamil",
    "ta": "tamil",
    "tamil": "tamil",

    # Hindi
    "hin": "hindi",
    "hin_Deva": "hindi",
    "hi": "hindi",
    "hindi": "hindi",

    # Bengali
    "ben": "bengali",
    "ben_Beng": "bengali",
    "bn": "bengali",
    "bengali": "bengali",

    # Arabic
    "arb": "arabic",
    "arb_Arab": "arabic",
    "ar": "arabic",
    "arabic": "arabic",

    # French
    "fra": "french",
    "fra_Latn": "french",
    "fr": "french",
    "french": "french",

    # German
    "deu": "german",
    "deu_Latn": "german",
    "de": "german",
    "german": "german",
}


# ------------------------------------------------------------
# 5. NLLB model prediction mapping
# ------------------------------------------------------------

MODEL_LABEL_TO_NAME = {

    "__label__sin_Sinh": "sinhala",

    "__label__pli_Sinh": "pali",

    # IMPORTANT:
    # Sinhala-script Sanskrit
    "__label__san_Sinh": "sanskrit",

    # Existing Devanagari Sanskrit
    "__label__san_Deva": "sanskrit_deva",

    "__label__eng_Latn": "english",
    "__label__tam_Taml": "tamil",
    "__label__hin_Deva": "hindi",
    "__label__ben_Beng": "bengali",
    "__label__arb_Arab": "arabic",
    "__label__fra_Latn": "french",
    "__label__deu_Latn": "german",
}


# ------------------------------------------------------------
# 6. Correctly distinguish Sanskrit-Sinh and Sanskrit-Deva
# ------------------------------------------------------------

def map_all_label(row):

    label = str(
        row.get("label", "")
    ).strip()

    source = str(
        row.get("source", "")
    ).strip()

    # Our project sources contain Sanskrit
    # written using SINHALA SCRIPT.
    if label in ["san", "sanskrit", "san_Sinh"]:

        if source in [
            "DCS",
            "SansinNT",
            "SiDiaC-v2"
        ]:
            return "sanskrit"

        # Other benchmark Sanskrit is Devanagari
        return "sanskrit_deva"

    # Explicit Devanagari label
    if label == "san_Deva":
        return "sanskrit_deva"

    return LABEL_MAPPING_ALL.get(label)


# ------------------------------------------------------------
# 7. Load benchmark
# ------------------------------------------------------------

def load_all_benchmark(file_path):

    records = []

    with open(
        file_path,
        encoding="utf-8"
    ) as f:

        for line in f:

            row = json.loads(line)

            mapped = map_all_label(row)

            if mapped in ALL_BENCHMARK_LANGUAGES:

                row["target_label"] = mapped
                records.append(row)

    return pd.DataFrame(records)


# ------------------------------------------------------------
# 8. Predict
# ------------------------------------------------------------

def predict_texts(texts):

    clean_texts = [
        str(text)
        .replace("\n", " ")
        .replace("\r", " ")
        .strip()
        for text in texts
    ]

    predictions, probabilities = model.predict(
        clean_texts,
        k=1
    )

    mapped_predictions = []

    for prediction in predictions:

        raw_label = prediction[0]

        mapped = MODEL_LABEL_TO_NAME.get(
            raw_label,
            "other"
        )

        mapped_predictions.append(mapped)

    return mapped_predictions


# ------------------------------------------------------------
# 9. Find benchmark files
# ------------------------------------------------------------

benchmark_files = sorted(
    glob.glob(
        os.path.join(
            benchmark_input_dir,
            "*.jsonl"
        )
    )
)

print("\nBenchmark files:")

for file_path in benchmark_files:
    print(" -", os.path.basename(file_path))


# ------------------------------------------------------------
# 10. Evaluate
# ------------------------------------------------------------

all_language_summary = []

per_language_summary = []


for file_path in benchmark_files:

    dataset_name = os.path.splitext(
        os.path.basename(file_path)
    )[0]

    df_all = load_all_benchmark(
        file_path
    )

    if df_all.empty:

        print(
            f"\nNo matching rows in "
            f"{dataset_name}"
        )

        continue


    print("\n")
    print("=" * 75)
    print(
        f"NLLB LID-220 FINETUNED "
        f"- {dataset_name}"
    )
    print("=" * 75)

    print(
        f"Samples: {len(df_all)}"
    )

    print("\nTrue-label counts:")

    print(
        df_all["target_label"]
        .value_counts()
        .reindex(
            ALL_BENCHMARK_LANGUAGES,
            fill_value=0
        )
    )


    # -----------------------
    # Predictions
    # -----------------------

    results = df_all.copy()

    results["predicted_label"] = (
        predict_texts(
            results["text"].tolist()
        )
    )

    y_true = results["target_label"]
    y_pred = results["predicted_label"]


    # -----------------------
    # Overall metrics
    # -----------------------

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=ALL_BENCHMARK_LANGUAGES,
        average="macro",
        zero_division=0
    )


    print("\n" + "-" * 75)

    print(
        f"Accuracy : {accuracy:.4f}"
    )

    print(
        f"Macro F1 : {macro_f1:.4f}"
    )

    print("-" * 75)


    # -----------------------
    # Per-language report
    # -----------------------

    print(
        "\nPer-language F1 scores:\n"
    )

    report = classification_report(
        y_true,
        y_pred,
        labels=ALL_BENCHMARK_LANGUAGES,
        output_dict=True,
        zero_division=0
    )


    for language in ALL_BENCHMARK_LANGUAGES:

        support = int(
            report[language]["support"]
        )

        f1 = report[language]["f1-score"]

        if support == 0:

            print(
                f"{language:16s} : "
                f"N/A (support=0)"
            )

        else:

            print(
                f"{language:16s} : "
                f"{f1:.4f} "
                f"(support={support})"
            )


        per_language_summary.append({

            "dataset": dataset_name,

            "language": language,

            "f1": (
                f1
                if support > 0
                else None
            ),

            "support": support
        })


    # -----------------------
    # Full classification report
    # -----------------------

    print(
        "\nFull classification report:\n"
    )

    print(
        classification_report(
            y_true,
            y_pred,
            labels=ALL_BENCHMARK_LANGUAGES,
            digits=4,
            zero_division=0
        )
    )


    # -----------------------
    # Save predictions
    # -----------------------

    output_csv = os.path.join(
        benchmark_output_dir,
        f"nllb_lid220_ft_{dataset_name}.csv"
    )

    results.to_csv(
        output_csv,
        index=False
    )

    print(
        "Saved predictions to:",
        output_csv
    )


    all_language_summary.append({

        "dataset": dataset_name,

        "rows": len(results),

        "accuracy": accuracy,

        "macro_f1": macro_f1
    })


# ============================================================
# 11. FINAL TABLE — directly usable for your spreadsheet
# ============================================================

per_language_df = pd.DataFrame(
    per_language_summary
)

spreadsheet_table = (
    per_language_df
    .pivot(
        index="dataset",
        columns="language",
        values="f1"
    )
    .reindex(
        columns=ALL_BENCHMARK_LANGUAGES
    )
    .round(4)
)


print("\n")
print("=" * 100)
print("FINAL PER-LANGUAGE F1 TABLE")
print("=" * 100)

display(spreadsheet_table)


# Overall summary
overall_summary_df = pd.DataFrame(
    all_language_summary
)

print("\nOVERALL RESULTS")

display(overall_summary_df)

Loaded model: D:\Projects\ML Projects\LangID - DSE project\models\nllb_lid_220_ft_e3_lr001.bin
Number of labels: 220

Benchmark files:
 - commonlid.jsonl
 - flores_plus.jsonl
 - wili-2018.jsonl


NLLB LID-220 FINETUNED - commonlid
Samples: 77974

True-label counts:
target_label
sinhala           2693
pali              3027
sanskrit          1327
sanskrit_deva      895
english          27461
tamil               81
hindi             3666
bengali           1886
arabic           26152
french            3233
german            7553
Name: count, dtype: int64

---------------------------------------------------------------------------
Accuracy : 0.9165
Macro F1 : 0.9492
---------------------------------------------------------------------------

Per-language F1 scores:

sinhala          : 0.9625 (support=2693)
pali             : 0.9500 (support=3027)
sanskrit         : 0.9053 (support=1327)
sanskrit_deva    : 0.9310 (support=895)
english          : 0.9072 (support=27461)
tamil            : 0.9

language,sinhala,pali,sanskrit,sanskrit_deva,english,tamil,hindi,bengali,arabic,french,german
dataset,,,,,,,,,,,
commonlid,0.9625,0.9500,0.9053,0.9310,0.9072,0.9875,0.9588,0.9874,0.9919,0.9192,0.9407
flores_plus,0.9760,0.9526,0.9053,0.9915,1.0000,1.0000,0.9921,1.0000,0.6667,0.9995,0.9995
wili-2018,0.9760,0.9526,0.9053,0.9909,0.9455,0.9940,0.9879,0.9429,NaN,0.9900,0.9899



OVERALL RESULTS


,dataset,rows,accuracy,macro_f1
0,commonlid,77974,0.916549,0.949225
1,flores_plus,16155,0.915135,0.953022
2,wili-2018,14047,0.961842,0.879547
